In [ ]:

# -------- INSTALL (run once) --------
!pip install -q langchain langchain-community faiss-cpu sentence-transformers

# -------- IMPORTS --------
import re
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

# -------- EMAIL VALIDATION --------
def is_valid_email(email):
    pattern = r'^[\w\.-]+@([\w\.-]+)\.\w+$'
    match = re.match(pattern, email)

    if not match:
        return False

    domain = email.split("@")[1].lower()

    valid_domains = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com"]

    return domain in valid_domains

# -------- CREATE KNOWLEDGE FILE --------
with open("knowledge.md", "w") as f:
    f.write("""
AutoStream Pricing:

Basic Plan:
$29/month
10 videos/month
720p resolution

Pro Plan:
$79/month
Unlimited videos
4K resolution
AI captions

Policies:
No refunds after 7 days
24/7 support only for Pro plan
""")

# -------- INTENT DETECTION --------
def detect_intent(text):
    text = text.lower()

    if any(word in text for word in ["buy", "start", "try", "subscribe", "purchase", "want"]):
        return "high_intent"
    elif any(word in text for word in ["price", "pricing", "plan", "plans", "cost"]):
        return "pricing"
    elif any(word in text for word in ["hi", "hello"]):
        return "greeting"

    return "general"

# -------- LEAD TOOL --------
def mock_lead_capture(name, email, platform):
    print(f"Lead captured: {name}, {email}, {platform}")

# -------- RAG SETUP --------
loader = TextLoader("knowledge.md")
docs = loader.load()

splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.from_documents(docs, embeddings)

def retrieve(query):
    return db.similarity_search(query, k=2)

# -------- RESPONSE --------
def generate_response(query, docs):
    context = "\n".join([d.page_content for d in docs])
    return f"""Here are our plans:

{context}

Let me know if you'd like to get started!
"""

# -------- STATE --------
state = {
    "messages": [],
    "intent": "",
    "name": None,
    "email": None,
    "platform": None
}

# -------- CHAT FUNCTION --------
def chat(msg):
    state["messages"].append(msg)

    # 🔥 Lead flow
    if state.get("intent") == "high_intent":

        if not state.get("name"):
            state["name"] = msg
            return "Please enter your email"

        elif not state.get("email"):
            # ✅ FIXED VALIDATION (INDENT CORRECT)
            if not is_valid_email(msg):
                return "❌ Please enter a valid email (example: name@gmail.com)"

            state["email"] = msg
            return "Which platform do you use? (YouTube/Instagram)"

        elif not state.get("platform"):
            state["platform"] = msg
            mock_lead_capture(state["name"], state["email"], state["platform"])
            return "🎉 Lead captured successfully!"

    # 🔥 Normal flow
    intent = detect_intent(msg)

    if intent == "greeting":
        return "Hi! 👋 How can I help you with AutoStream?"

    elif intent == "pricing":
        docs = retrieve(msg)
        return generate_response(msg, docs)

    elif intent == "high_intent":
        state["intent"] = "high_intent"
        state["name"] = ""
        state["email"] = ""
        state["platform"] = ""
        return "Great! Let's get started. What's your name?"

    return "I'm here to help! 😊"

In [ ]:

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Bot: Bye 👋")
        break

    response = chat(user_input)
    print("Bot:", response)